# Project RIPRA (ऋप्र): Shack-Hartmann Wavefront Reconstruction & Turbulence Characterization
This notebook runs the data generation, model definition, and training pipeline in Kaggle/Colab with GPU acceleration.

### Objectives:
1. Generate a massive synthetic Kolmogorov dataset based on physical Shack-Hartmann parameters.
2. Train a Fully Connected Network (MLP) as a baseline.
3. Train a Spatial 2D ResNet CNN mapping displacements to Zernike coefficients.

## Step 1: Write All Configuration, Spot Coordinates, and Python Files to Disk
We write all configuration files and scripts directly to the Kaggle filesystem via Base64 decoding (to avoid quote/triple-quote nesting syntax errors).

In [ ]:
import os
import base64

# Create required directory structures
os.makedirs('config', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('data_ai', exist_ok=True)
os.makedirs('ml_checkpoints', exist_ok=True)

conf_b64 = 'IyBSSVBQQSBzeXN0ZW0gY29uZmlndXJhdGlvbgojIEFsbCB2YWx1ZXMgd2l0aCBTSSB1bml0cyB1bmxlc3Mgbm90ZWQuCiMgTGluZXMgc3RhcnRpbmcgd2l0aCAjIGFyZSBjb21tZW50cy4gVW5rbm93biBrZXlzIGFyZSBpZ25vcmVkLgoKIyAtLS0gQ2FtZXJhIC0tLQpjYW1lcmFfcGl4c2l6ZSA9IDcuNGUtNiAgICAgICAjIHBpeGVsIHNpemUgKG0pCmZyYW1lX3dpZHRoICAgID0gNjQ4ICAgICAgICAgICAjIGRldGVjdG9yIHdpZHRoIChwaXhlbHMpCmZyYW1lX2hlaWdodCAgID0gNDkyICAgICAgICAgICAjIGRldGVjdG9yIGhlaWdodCAocGl4ZWxzKQoKIyAtLS0gTWljcm9sZW5zIEFycmF5IChNTEEpIC0tLQp0b3RsZW5zZXMgICAgICA9IDEyNyAgICAgICAgICAjIG51bWJlciBvZiBsZW5zbGV0cwpmbGVuZ3RoICAgICAgICA9IDE4ZS0zICAgICAgICAjIGZvY2FsIGxlbmd0aCAobSkKcGl0Y2ggICAgICAgICAgPSAzMDBlLTYgICAgICAgIyBsZW5zbGV0IHBpdGNoIChtKQpzYV9yYWRpdXMgICAgICA9IDE1MGUtNiAgICAgICAjIHN1Yi1hcGVydHVyZSByYWRpdXMgPSBwaXRjaC8yIChtKQoKIyAtLS0gUHVwaWwgLS0tCnB1cGlsX3JhZGl1cyAgID0gMmUtMyAgICAgICAgICMgcHVwaWwgcmFkaXVzIChtKSwgZGlhbWV0ZXIgPSA0IG1tCgojIC0tLSBXYXZlbGVuZ3RoIC0tLQp3YXZlbGVuZ3RoICAgICA9IDYzMi44ZS05ICAgICAjIEhlTmUgbGFzZXIgKG0pCgojIC0tLSBDZW50cm9pZGluZyAtLS0KdGhyZXNoX2JpbmFyeSAgPSAwLjA4ICAgICAgICAgIyBiaW5hcnkgaW1hZ2UgdGhyZXNob2xkIChyZWxhdGl2ZSkKY2VudHJvaWRfcGVyY2VudCA9IDAuMiAgICAgICAgICMgcmVsYXRpdmUgdGhyZXNob2xkIGZvciBDb0cKY29hcnNlX2dyaWRfcmFkaXVzID0gMTIgICAgICAgICMgcGl4ZWxzLCBjb2Fyc2UgZ3JpZCBoYWxmLXdpbmRvdwoKIyAtLS0gWmVybmlrZSBtb2RhbCByZWNvbnN0cnVjdGlvbiAtLS0KemVybmlrZV9ubWF4ICAgPSA1ICAgICAgICAgICAgICMgbWF4IHJhZGlhbCBvcmRlciAoTm9sbCBpbmRleGluZykKCiMgLS0tIERlZm9ybWFibGUgTWlycm9yIC0tLQpkbV9uYWN0X3ggICAgICA9IDEyICAgICAgICAgICAgIyBhY3R1YXRvcnMgYWNyb3NzIFgKZG1fbmFjdF95ICAgICAgPSAxMiAgICAgICAgICAgICMgYWN0dWF0b3JzIGFjcm9zcyBZCmNvdXBsaW5nICAgICAgID0gMC4xNSAgICAgICAgICAjIGludGVyLWFjdHVhdG9yIGNvdXBsaW5nIGNvZWZmaWNpZW50Cg=='
with open('config/system.conf', 'wb') as f:
    f.write(base64.b64decode(conf_b64))

spots_b64 = 'U3BvdF9JRCxyZWZfY3gscmVmX2N5LGNvbF9taW4sY29sX21heCxyb3dfbWluLHJvd19tYXgNCjAsMzM2LjIzNTgsMzUuNjA1NSwzMjIsMzUwLDIxLDQ5DQoxLDI5NS4yNzU1LDM1Ljk1MzQsMjgxLDMwOSwyMiw1MA0KMiwzNzcuMDMyOSwzNS42OTY4LDM2MywzOTEsMjIsNTANCjMsMjU0LjUzMzksMzYuNTM5MiwyNDAsMjY4LDIyLDUwDQo0LDQxNy4yMzQ5LDM1LjQzNjUsNDAzLDQzMSwyMSw0OQ0KNSw0NTguMzAzOSwzNS4xMDA5LDQ0NCw0NzIsMjEsNDkNCjYsMjE0LjMxOTgsMzYuMjk3NywyMDAsMjI4LDIyLDUwDQo3LDMxNS45MDMyLDcxLjIxNDcsMzAyLDMzMCw1Niw4NA0KOCwzNTYuODcxOSw3MS4wNjgzLDM0MywzNzEsNTYsODQNCjksMzk3LjYzNjcsNzEuMDA5MSwzODMsNDExLDU2LDg0DQoxMCw0MzcuNzM4Niw3MC45MDU1LDQyMyw0NTEsNTYsODQNCjExLDQ3OC40OTM3LDcwLjY0MDMsNDY0LDQ5Miw1Niw4NA0KMTIsMTkzLjU2NTksNzEuNTMyNCwxNzksMjA3LDU3LDg1DQoxMywyNzUuMTYwMiw3MS4xODMwLDI2MSwyODksNTcsODUNCjE0LDIzNC44MTk1LDcxLjg2MDYsMjIwLDI0OCw1Nyw4NQ0KMTUsMzc3LjYwODYsMTA2LjQwMTksMzYzLDM5MSw5MiwxMjANCjE2LDQxNy42MzkwLDEwNi4wNTExLDQwMyw0MzEsOTEsMTE5DQoxNyw0OTkuMTkwOSwxMDUuNzk1Myw0ODUsNTEzLDkxLDExOQ0KMTgsMTczLjM0OTksMTA2LjU4ODAsMTU5LDE4Nyw5MiwxMjANCjE5LDIxNC42MTgyLDEwNi4zMDI5LDIwMCwyMjgsOTIsMTIwDQoyMCwyNTQuODExOCwxMDYuNTQzMCwyNDAsMjY4LDkyLDEyMA0KMjEsMjk1LjMwMjUsMTA2LjkwNzQsMjgxLDMwOSw5MywxMjENCjIyLDMzNi42Nzg1LDEwNi44NjAyLDMyMywzNTEsOTMsMTIxDQoyMyw0NTguODM1NywxMDYuMDU1Miw0NDUsNDczLDkyLDEyMA0KMjQsMzk3LjY1NTEsMTQxLjI1MzAsMzgzLDQxMSwxMjcsMTU1DQoyNSw1MTkuNTUwNSwxNDAuNTc3MSw1MDUsNTMzLDEyNiwxNTQNCjI2LDMxNS42NzUyLDE0Mi4zOTgxLDMwMSwzMjksMTI4LDE1Ng0KMjcsNDM4LjE5NjYsMTQxLjI5MzMsNDI0LDQ1MiwxMjcsMTU1DQoyOCw0NzguNjgwNiwxNDEuMTE4NCw0NjQsNDkyLDEyNywxNTUNCjI5LDE5NC4xODQwLDE0Mi4xNDQ1LDE4MCwyMDgsMTI3LDE1NQ0KMzAsMjM1LjEwMjAsMTQxLjg2MjQsMjIxLDI0OSwxMjgsMTU2DQozMSwzNTYuODIyNiwxNDEuODMyNSwzNDIsMzcwLDEyOCwxNTYNCjMyLDE1My4xNTY0LDE0MS43MDg5LDEzOSwxNjcsMTI3LDE1NQ0KMzMsMjc1LjQyMTIsMTQyLjE4OTEsMjYxLDI4OSwxMjgsMTU2DQozNCw0MTcuNzY4OSwxNzYuNDg5MSw0MDMsNDMxLDE2MiwxOTANCjM1LDM3Ni44NzMxLDE3Ni44NTU0LDM2MywzOTEsMTYyLDE5MA0KMzYsMjU1LjE3ODMsMTc3LjMwNjgsMjQwLDI2OCwxNjMsMTkxDQozNywyOTUuODcwNSwxNzcuMjY5NSwyODEsMzA5LDE2MywxOTENCjM4LDMzNi4zNDc1LDE3Ny4xOTMyLDMyMiwzNTAsMTYyLDE5MA0KMzksNDU4LjM3MTEsMTc2LjMyMDQsNDQ0LDQ3MiwxNjIsMTkwDQo0MCw0OTkuMDQ5NCwxNzYuMDY1Miw0ODUsNTEzLDE2MSwxODkNCjQxLDU0MC4wMzAzLDE3Ni4xMzA5LDUyNSw1NTMsMTYyLDE5MA0KNDIsMTc0LjEwMDcsMTc3LjE5NjEsMTYwLDE4OCwxNjMsMTkxDQo0MywyMTQuNjUyMCwxNzcuMTcwMCwyMDAsMjI4LDE2MywxOTENCjQ0LDEzMi44NTUwLDE3Ny4wMjMyLDExOCwxNDYsMTYyLDE5MA0KNDUsMTk0LjY0NDYsMjEyLjIxOTQsMTgwLDIwOCwxOTcsMjI1DQo0NiwzNTYuODM2NiwyMTIuMDcyNCwzNDIsMzcwLDE5OCwyMjYNCjQ3LDM5Ny40MDc3LDIxMi4wMTg3LDM4Myw0MTEsMTk3LDIyNQ0KNDgsNTYwLjI1MTYsMjExLjI3MjQsNTQ2LDU3NCwxOTcsMjI1DQo0OSwyMzUuNjc3NSwyMTIuMjQ1MywyMjEsMjQ5LDE5NywyMjUNCjUwLDI3NS45MjA3LDIxMi4yNTE0LDI2MSwyODksMTk4LDIyNg0KNTEsMzE2LjA2MTUsMjEyLjAwOTgsMzAyLDMzMCwxOTcsMjI1DQo1Miw0MzcuOTc0MSwyMTEuOTU2OCw0MjMsNDUxLDE5OCwyMjYNCjUzLDQ3OS4wNTg5LDIxMS40NjM1LDQ2NSw0OTMsMTk3LDIyNQ0KNTQsNTE5LjI4NzMsMjExLjQ1MjAsNTA1LDUzMywxOTcsMjI1DQo1NSwxMTIuNzE0MywyMTIuMTA3NSw5OCwxMjYsMTk3LDIyNQ0KNTYsMTUzLjg0MTcsMjEyLjcwMTksMTM5LDE2NywxOTgsMjI2DQo1Nyw0MTcuOTA0NCwyNDYuODg1Miw0MDQsNDMyLDIzMiwyNjANCjU4LDI1NS45MTY4LDI0Ny4xNzg1LDI0MSwyNjksMjMyLDI2MA0KNTksMzM2LjYwOTIsMjQ3LjMzNTUsMzIyLDM1MCwyMzMsMjYxDQo2MCw0NTguNjAxMiwyNDYuOTAxMCw0NDMsNDcxLDIzMiwyNjANCjYxLDU4MC42MzMxLDI0Ny4wODYyLDU2Niw1OTQsMjMzLDI2MQ0KNjIsMjE1LjI0MTUsMjQ3LjM0NDEsMjAwLDIyOCwyMzMsMjYxDQo2MywyOTUuODU3OCwyNDcuNDUyNCwyODEsMzA5LDIzMywyNjENCjY0LDM3Ny4wNjgwLDI0Ny4xNTQ1LDM2MiwzOTAsMjMzLDI2MQ0KNjUsNDk5LjE0ODksMjQ3LjA4NDAsNDg0LDUxMiwyMzMsMjYxDQo2Niw1MzkuNjk4NiwyNDYuOTI5MCw1MjUsNTUzLDIzMiwyNjANCjY3LDEzMy43MTE5LDI0Ny42MjgxLDExOSwxNDcsMjMzLDI2MQ0KNjgsMTc0LjE2NDQsMjQ3LjQ0NTMsMTYwLDE4OCwyMzMsMjYxDQo2OSw5Mi45ODMzLDI0OC40MTA5LDc5LDEwNywyMzQsMjYyDQo3MCwzNTYuOTg5MSwyODEuOTExMywzNDMsMzcxLDI2NywyOTUNCjcxLDM5Ny44NTg3LDI4Mi4yMzEwLDM4Myw0MTEsMjY4LDI5Ng0KNzIsNDM4LjQ5OTAsMjgyLjI1ODcsNDI0LDQ1MiwyNjgsMjk2DQo3MywxOTQuODgwMSwyODIuNjE4NywxODEsMjA5LDI2OCwyOTYNCjc0LDMxNi4xMDcwLDI4Mi4xMjgxLDMwMSwzMjksMjY3LDI5NQ0KNzUsNDc5LjExMzIsMjgyLjAwNjQsNDY0LDQ5MiwyNjgsMjk2DQo3Niw1MTkuNzY0MCwyODEuODY2MCw1MDUsNTMzLDI2NywyOTUNCjc3LDU2MC44Njk0LDI4MS44OTc3LDU0Niw1NzQsMjY3LDI5NQ0KNzgsMTUzLjk3MTAsMjgyLjgzNzYsMTM5LDE2NywyNjgsMjk2DQo3OSwyMzUuOTE0NSwyODIuNjI3OSwyMjEsMjQ5LDI2OCwyOTYNCjgwLDI3NS44NDgyLDI4Mi43MTIxLDI2MCwyODgsMjY4LDI5Ng0KODEsMTEyLjc4MTEsMjgzLjU2NjcsOTksMTI3LDI2OSwyOTcNCjgyLDMzNi44MTc0LDMxNy4xMjA3LDMyMiwzNTAsMzAyLDMzMA0KODMsMjk2LjIwODcsMzE3LjM5NTYsMjgxLDMwOSwzMDMsMzMxDQo4NCwzNzcuODI2OSwzMTcuMTg0MSwzNjMsMzkxLDMwMywzMzENCjg1LDQxOC42MDM3LDMxNy44NTA5LDQwNCw0MzIsMzAzLDMzMQ0KODYsNDk5LjQwODcsMzE3LjAxOTAsNDg1LDUxMywzMDMsMzMxDQo4Nyw1NDAuMjc1OSwzMTcuMTM2Niw1MjUsNTUzLDMwMiwzMzANCjg4LDE3NC4zODU0LDMxOC4wNjY2LDE2MCwxODgsMzAzLDMzMQ0KODksMjE1LjI1OTAsMzE3Ljg4NjMsMjAxLDIyOSwzMDMsMzMxDQo5MCwyNTUuOTM5OCwzMTcuNjU4NywyNDEsMjY5LDMwMywzMzENCjkxLDQ1OC44ODIxLDMxNy41MTQxLDQ0NCw0NzIsMzAzLDMzMQ0KOTIsMTMzLjU4MDAsMzE4LjA3NzQsMTE5LDE0NywzMDQsMzMyDQo5Myw0NzkuMjk4NiwzNTIuNDE0Nyw0NjUsNDkzLDMzOCwzNjYNCjk0LDM1Ny42NDg1LDM1My4wNzcyLDM0MywzNzEsMzM5LDM2Nw0KOTUsNDM4LjYyNTgsMzUyLjcwODQsNDI0LDQ1MiwzMzgsMzY2DQo5NiwyMzUuMzk1NiwzNTMuMDkxMCwyMjAsMjQ4LDMzOSwzNjcNCjk3LDI3Ni4xMzUyLDM1My4wODkxLDI2MiwyOTAsMzM5LDM2Nw0KOTgsMzE2LjQ5MDEsMzUzLjA2MzIsMzAxLDMyOSwzMzksMzY3DQo5OSwzOTguMTk5NSwzNTMuMDgyMSwzODQsNDEyLDMzOSwzNjcNCjEwMCw1MjAuMDI1NCwzNTIuNTUyMSw1MDUsNTMzLDMzOCwzNjYNCjEwMSwxNTMuNzM1MywzNTMuNTQ4MSwxMzksMTY3LDMzOSwzNjcNCjEwMiwxOTQuODI5OSwzNTMuMjA3OSwxODAsMjA4LDMzOSwzNjcNCjEwMywzNzguMjI3MCwzODguMDE0MSwzNjMsMzkxLDM3Myw0MDENCjEwNCwzMzcuMjMxMSwzODguMDQ4NSwzMjMsMzUxLDM3Myw0MDENCjEwNSw0NTkuMDEzMSwzODcuNTM5Nyw0NDUsNDczLDM3Myw0MDENCjEwNiwyOTYuNjI0MiwzODguMDUzMywyODIsMzEwLDM3Myw0MDENCjEwNyw0MTguNTU1MCwzODcuOTY4MSw0MDQsNDMyLDM3Myw0MDENCjEwOCwxNzQuOTA2MiwzODcuNjUyOCwxNjAsMTg4LDM3Myw0MDENCjEwOSwyNTUuNzg5MCwzODguMDM4NSwyNDEsMjY5LDM3NCw0MDINCjExMCw0OTkuNzE0MCwzODcuNzIwMSw0ODUsNTEzLDM3Myw0MDENCjExMSwyMTUuMjY2NiwzODguMjk1NSwyMDEsMjI5LDM3NCw0MDINCjExMiwzMTcuMzY3OSw0MjMuMjAwMCwzMDMsMzMxLDQwOSw0MzcNCjExMywzNTcuODc4NCw0MjMuMDExOCwzNDMsMzcxLDQwOSw0MzcNCjExNCwzOTguMzc4Nyw0MjIuODY5MCwzODQsNDEyLDQwOCw0MzYNCjExNSw0MzguODA0OSw0MjIuODgwOCw0MjQsNDUyLDQwOSw0MzcNCjExNiw0NzkuNjI2OSw0MjIuNzYzNiw0NjUsNDkzLDQwOCw0MzYNCjExNywyMzYuMDc2NSw0MjMuMzU4OSwyMjIsMjUwLDQwOSw0MzcNCjExOCwyNzYuMTg2Myw0MjMuMjgxNywyNjEsMjg5LDQwOSw0MzcNCjExOSwxOTQuOTQ3MCw0MjQuMDU2MiwxODAsMjA4LDQxMCw0MzgNCjEyMCwzNzguMTU1OCw0NTguOTU3MywzNjMsMzkxLDQ0NSw0NzMNCjEyMSw0MTguNzQyMiw0NTguNDMzMSw0MDQsNDMyLDQ0NCw0NzINCjEyMiwyNTYuMzYzNyw0NTguOTQxNywyNDIsMjcwLDQ0NCw0NzINCjEyMywyOTYuNzY4Niw0NTkuMDM2MiwyODIsMzEwLDQ0NCw0NzINCjEyNCwzMzcuNTg3OSw0NTguNjAyNywzMjMsMzUxLDQ0NSw0NzMNCjEyNSw0NTkuMzgzNiw0NTguMzM4OSw0NDQsNDcyLDQ0NCw0NzINCjEyNiwyMTUuMjc2OSw0NTkuMjg5NywyMDEsMjI5LDQ0NSw0NzMNCg=='
with open('results/reference_centroids_c.csv', 'wb') as f:
    f.write(base64.b64decode(spots_b64))

gen_b64 = 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKZ2VuZXJhdGVfZGF0YXNldC5weSAtIEdlbmVyYXRlIHN5bnRoZXRpYyBLb2xtb2dvcm92IHR1cmJ1bGVuY2UgZGF0YXNldCBmb3IgU2hhY2stSGFydG1hbm4gV0ZTCiIiIgppbXBvcnQgb3MKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgc2NpcHkuc3BlY2lhbCBhcyBzcGVjaWFsCmltcG9ydCBwYW5kYXMgYXMgcGQKCmRlZiBub2xsX3RvX25tKGopOgogICAgIiIiQ29udmVydCBOb2xsIGluZGV4IGogKDEtYmFzZWQpIHRvIHJhZGlhbCBvcmRlciBuIGFuZCBhemltdXRoYWwgZnJlcXVlbmN5IG0iIiIKICAgIGlmIGogPT0gMToKICAgICAgICByZXR1cm4gMCwgMAogICAgY3VycmVudF9qID0gMgogICAgZm9yIG4gaW4gcmFuZ2UoMSwgMTAwKToKICAgICAgICBmb3IgbSBpbiByYW5nZShuICUgMiwgbiArIDEsIDIpOgogICAgICAgICAgICBpZiBtID09IDA6CiAgICAgICAgICAgICAgICBpZiBjdXJyZW50X2ogPT0gajoKICAgICAgICAgICAgICAgICAgICByZXR1cm4gbiwgMAogICAgICAgICAgICAgICAgY3VycmVudF9qICs9IDEKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICMgU2FtZSBwYXJpdHkKICAgICAgICAgICAgICAgIGlmIGN1cnJlbnRfaiAlIDIgPT0gMToKICAgICAgICAgICAgICAgICAgICBpZiBjdXJyZW50X2ogPT0gajoKICAgICAgICAgICAgICAgICAgICAgICAgcmV0dXJuIG4sIC1tCiAgICAgICAgICAgICAgICAgICAgaWYgY3VycmVudF9qICsgMSA9PSBqOgogICAgICAgICAgICAgICAgICAgICAgICByZXR1cm4gbiwgbQogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICBpZiBjdXJyZW50X2ogPT0gajoKICAgICAgICAgICAgICAgICAgICAgICAgcmV0dXJuIG4sIG0KICAgICAgICAgICAgICAgICAgICBpZiBjdXJyZW50X2ogKyAxID09IGo6CiAgICAgICAgICAgICAgICAgICAgICAgIHJldHVybiBuLCAtbQogICAgICAgICAgICAgICAgY3VycmVudF9qICs9IDIKCmRlZiBub2xsX2NvdmFyaWFuY2UobjEsIG0xLCBqMSwgbjIsIG0yLCBqMik6CiAgICAiIiJDb21wdXRlIHRoZSBOb2xsIGNvdmFyaWFuY2UgY29lZmZpY2llbnQgQ19paiBmb3IgS29sbW9nb3JvdiB0dXJidWxlbmNlIiIiCiAgICBpZiBhYnMobTEpICE9IGFicyhtMik6CiAgICAgICAgcmV0dXJuIDAuMAogICAgIyBQYXJpdHkgY2hlY2s6IGJvdGggbXVzdCBiZSBldmVuIG9yIGJvdGggb2RkIE5vbGwgaW5kaWNlcyBmb3Igc2FtZSBtCiAgICBpZiAoajEgJSAyKSAhPSAoajIgJSAyKSBhbmQgYWJzKG0xKSAhPSAwOgogICAgICAgIHJldHVybiAwLjAKICAgIAogICAgIyBTaWduIGZhY3RvcgogICAgc2lnbiA9ICgtMS4wKSAqKiAoKG4xICsgbjIgLSAyICogYWJzKG0xKSkgLyAyKQogICAgCiAgICAjIEdhbW1hIHRlcm1zCiAgICBudW0gPSAoc2lnbiAqIG5wLnNxcnQoKG4xICsgMSkgKiAobjIgKyAxKSkgKiAKICAgICAgICAgICBzcGVjaWFsLmdhbW1hKDE0LjAgLyAzLjApICogCiAgICAgICAgICAgc3BlY2lhbC5nYW1tYSgobjEgKyBuMiAtIDUuMCAvIDMuMCkgLyAyLjApKQogICAgZGVuID0gKHNwZWNpYWwuZ2FtbWEoKG4xIC0gbjIgKyAxNy4wIC8gMy4wKSAvIDIuMCkgKiAKICAgICAgICAgICBzcGVjaWFsLmdhbW1hKChuMiAtIG4xICsgMTcuMCAvIDMuMCkgLyAyLjApICogCiAgICAgICAgICAgc3BlY2lhbC5nYW1tYSgobjEgKyBuMiArIDIzLjAgLyAzLjApIC8gMi4wKSkKICAgIAogICAgcmV0dXJuIG51bSAvIGRlbgoKZGVmIGdldF9ub2xsX2NvdmFyaWFuY2VfbWF0cml4KG1heF9qKToKICAgICIiIkNvbXB1dGUgdGhlIE5vbGwgY292YXJpYW5jZSBtYXRyaXggZm9yIGogPSAyIHRvIG1heF9qIChleGNsdWRpbmcgcGlzdG9uKSIiIgogICAgbm1vZGVzID0gbWF4X2ogLSAxCiAgICBDID0gbnAuemVyb3MoKG5tb2Rlcywgbm1vZGVzKSkKICAgIG1vZGVzID0gW10KICAgIGZvciBpZHggaW4gcmFuZ2Uobm1vZGVzKToKICAgICAgICBqID0gaWR4ICsgMgogICAgICAgIG4sIG0gPSBub2xsX3RvX25tKGopCiAgICAgICAgbW9kZXMuYXBwZW5kKChqLCBuLCBtKSkKICAgICAgICAKICAgIGZvciBpIGluIHJhbmdlKG5tb2Rlcyk6CiAgICAgICAgajEsIG4xLCBtMSA9IG1vZGVzW2ldCiAgICAgICAgZm9yIGpfaWR4IGluIHJhbmdlKG5tb2Rlcyk6CiAgICAgICAgICAgIGoyLCBuMiwgbTIgPSBtb2Rlc1tqX2lkeF0KICAgICAgICAgICAgQ1tpLCBqX2lkeF0gPSBub2xsX2NvdmFyaWFuY2UobjEsIG0xLCBqMSwgbjIsIG0yLCBqMikKICAgICAgICAgICAgCiAgICByZXR1cm4gQywgbW9kZXMKCiMgWmVybmlrZSByYWRpYWwgY29lZmZpY2llbnQgSGVscGVyCmRlZiB6ZXJuaWtlX2NvZWZmKG4sIG0sIHMpOgogICAgYWJzX20gPSBhYnMobSkKICAgIG51bSA9IHNwZWNpYWwuZmFjdG9yaWFsKG4gLSBzKQogICAgZGVuID0gc3BlY2lhbC5mYWN0b3JpYWwocykgKiBzcGVjaWFsLmZhY3RvcmlhbCgobiArIGFic19tKS8vMiAtIHMpICogc3BlY2lhbC5mYWN0b3JpYWwoKG4gLSBhYnNfbSkvLzIgLSBzKQogICAgc2lnbiA9IDEuMCBpZiBzICUgMiA9PSAwIGVsc2UgLTEuMAogICAgcmV0dXJuIHNpZ24gKiBudW0gLyBkZW4KCiMgRXZhbHVhdGUgWmVybmlrZSBkZXJpdmF0aXZlcyBhdCBub3JtYWxpemVkIHBvbGFyL2NhcnRlc2lhbiBjb29yZGluYXRlCmRlZiB6ZXJuaWtlX2Rlcml2YXRpdmVzKG4sIG0sIHgsIHkpOgogICAgcmhvID0gbnAuc3FydCh4KioyICsgeSoqMikKICAgIHRoZXRhID0gbnAuYXJjdGFuMih5LCB4KQogICAgYWJzX20gPSBhYnMobSkKICAgIAogICAgUiA9IDAuMAogICAgZFIgPSAwLjAKICAgIGZvciBzIGluIHJhbmdlKChuIC0gYWJzX20pIC8vIDIgKyAxKToKICAgICAgICBjID0gemVybmlrZV9jb2VmZihuLCBtLCBzKQogICAgICAgIHBvd19yaG8gPSBuIC0gMipzCiAgICAgICAgaWYgcG93X3JobyA+IDA6CiAgICAgICAgICAgIFIgKz0gYyAqIChyaG8gKiogcG93X3JobykKICAgICAgICAgICAgZFIgKz0gYyAqIHBvd19yaG8gKiAocmhvICoqIChwb3dfcmhvIC0gMSkpCiAgICAgICAgZWxpZiBwb3dfcmhvID09IDA6CiAgICAgICAgICAgIFIgKz0gYwogICAgICAgICAgICAKICAgIG5vcm1fZmFjdG9yID0gbnAuc3FydChuICsgMSkgaWYgbSA9PSAwIGVsc2UgbnAuc3FydCgyICogKG4gKyAxKSkKICAgIFIgKj0gbm9ybV9mYWN0b3IKICAgIGRSICo9IG5vcm1fZmFjdG9yCiAgICAKICAgIGNvc19tdCA9IG5wLmNvcyhhYnNfbSAqIHRoZXRhKQogICAgc2luX210ID0gbnAuc2luKGFic19tICogdGhldGEpCiAgICAKICAgIGlmIG0gPj0gMDoKICAgICAgICBkel9kcmhvID0gZFIgKiBjb3NfbXQKICAgICAgICBkel9kdGhldGEgPSAtYWJzX20gKiBSICogc2luX210CiAgICBlbHNlOgogICAgICAgIGR6X2RyaG8gPSBkUiAqIHNpbl9tdAogICAgICAgIGR6X2R0aGV0YSA9IGFic19tICogUiAqIGNvc19tdAogICAgICAgIAogICAgaWYgcmhvIDwgMWUtOToKICAgICAgICBpZiBuID09IDE6CiAgICAgICAgICAgIGlmIG0gPT0gMToKICAgICAgICAgICAgICAgIHJldHVybiBub3JtX2ZhY3RvciwgMC4wCiAgICAgICAgICAgIGVsaWYgbSA9PSAtMToKICAgICAgICAgICAgICAgIHJldHVybiAwLjAsIG5vcm1fZmFjdG9yCiAgICAgICAgcmV0dXJuIDAuMCwgMC4wCiAgICAgICAgCiAgICBkemR4ID0gZHpfZHJobyAqIG5wLmNvcyh0aGV0YSkgLSBkel9kdGhldGEgKiBucC5zaW4odGhldGEpIC8gcmhvCiAgICBkemR5ID0gZHpfZHJobyAqIG5wLnNpbih0aGV0YSkgKyBkel9kdGhldGEgKiBucC5jb3ModGhldGEpIC8gcmhvCiAgICByZXR1cm4gZHpkeCwgZHpkeQoKZGVmIGxvYWRfc3lzdGVtX2NvbmZpZyhjb25maWdfcGF0aCk6CiAgICBjZmcgPSB7fQogICAgd2l0aCBvcGVuKGNvbmZpZ19wYXRoLCAncicpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgbGluZSA9IGxpbmUuc3RyaXAoKQogICAgICAgICAgICBpZiBub3QgbGluZSBvciBsaW5lLnN0YXJ0c3dpdGgoJyMnKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGlmICc9JyBpbiBsaW5lOgogICAgICAgICAgICAgICAga2V5LCB2YWwgPSBsaW5lLnNwbGl0KCc9JywgMSkKICAgICAgICAgICAgICAgIGtleSA9IGtleS5zdHJpcCgpCiAgICAgICAgICAgICAgICB2YWwgPSB2YWwuc3BsaXQoJyMnLCAxKVswXS5zdHJpcCgpCiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgY2ZnW2tleV0gPSBmbG9hdCh2YWwpIGlmICcuJyBpbiB2YWwgb3IgJ2UnIGluIHZhbCBlbHNlIGludCh2YWwpCiAgICAgICAgICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICAgICAgICAgICAgICBjZmdba2V5XSA9IHZhbAogICAgcmV0dXJuIGNmZwoKZGVmIG1haW4oKToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJHZW5lcmF0ZSBzeW50aGV0aWMgU2hhY2stSGFydG1hbm4gZGF0YXNldCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNhbXBsZXMiLCB0eXBlPWludCwgZGVmYXVsdD0xMDAwMCwgaGVscD0iTnVtYmVyIG9mIHNhbXBsZXMgdG8gZ2VuZXJhdGUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NDIsIGhlbHA9IlJhbmRvbSBzZWVkIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbm9pc2UiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMSwgaGVscD0iU3RhbmRhcmQgZGV2aWF0aW9uIG9mIGRpc3BsYWNlbWVudCBub2lzZSBpbiBwaXhlbHMiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXQiLCB0eXBlPXN0ciwgZGVmYXVsdD0iZGF0YV9haS9kYXRhc2V0Lm5weiIsIGhlbHA9Ik91dHB1dCBmaWxlcGF0aCIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgCiAgICBucC5yYW5kb20uc2VlZChhcmdzLnNlZWQpCiAgICAKICAgICMgMS4gTG9hZCBjb25maWcgYW5kIHNwb3QgY29vcmRpbmF0ZXMKICAgIHByaW50KCJMb2FkaW5nIHN5c3RlbSBjb25maWd1cmF0aW9uLi4uIikKICAgIGNmZyA9IGxvYWRfc3lzdGVtX2NvbmZpZygiY29uZmlnL3N5c3RlbS5jb25mIikKICAgIAogICAgc3BvdHNfZmlsZSA9ICJyZXN1bHRzL3JlZmVyZW5jZV9jZW50cm9pZHNfYy5jc3YiCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoc3BvdHNfZmlsZSk6CiAgICAgICAgcHJpbnQoZiJFcnJvcjoge3Nwb3RzX2ZpbGV9IG5vdCBmb3VuZC4gUGxlYXNlIHJ1biB0ZXN0X2NlbnRyb2lkIGZpcnN0IHRvIGNhbGlicmF0ZS4iKQogICAgICAgIHJldHVybgogICAgICAgIAogICAgc3BvdHNfZGYgPSBwZC5yZWFkX2NzdihzcG90c19maWxlKQogICAgbnNwb3RzID0gbGVuKHNwb3RzX2RmKQogICAgcHJpbnQoZiJMb2FkZWQge25zcG90c30gYWN0aXZlIHN1Yi1hcGVydHVyZSBjb29yZGluYXRlcy4iKQogICAgCiAgICAjIDIuIFNldHVwIFplcm5pa2UgY292YXJpYW5jZQogICAgemVybmlrZV9ubWF4ID0gaW50KGNmZ1siemVybmlrZV9ubWF4Il0pCiAgICBtYXhfaiA9ICh6ZXJuaWtlX25tYXggKyAxKSAqICh6ZXJuaWtlX25tYXggKyAyKSAvLyAyCiAgICBubW9kZXMgPSBtYXhfaiAtIDEgIyBleGNsdWRlIHBpc3RvbgogICAgcHJpbnQoZiJTZXR0aW5nIHVwIFplcm5pa2UgY292YXJpYW5jZSBmb3Ige25tb2Rlc30gbW9kZXMgKE5vbGwgcmFkaWFsIG9yZGVyIDw9IHt6ZXJuaWtlX25tYXh9KS4uLiIpCiAgICAKICAgIEMsIG1vZGVzID0gZ2V0X25vbGxfY292YXJpYW5jZV9tYXRyaXgobWF4X2opCiAgICAKICAgICMgMy4gUHJlY29tcHV0ZSBaZXJuaWtlIERlcml2YXRpdmUgTWF0cml4IChacHJpbWUpIHZpYSBhcmVhIGludGVncmF0aW9uCiAgICBwcmludCgiUHJlY29tcHV0aW5nIFplcm5pa2UgZGVyaXZhdGl2ZSBtYXRyaXggKFpwcmltZSkuLi4iKQogICAgTSA9IDE1ICMgMTV4MTUgcXVhZHJhdHVyZSBncmlkCiAgICByYmFyID0gY2ZnWyJzYV9yYWRpdXMiXSAvIGNmZ1sicHVwaWxfcmFkaXVzIl0KICAgIGtrID0gY2ZnWyJwdXBpbF9yYWRpdXMiXSAvIChucC5waSAqIGNmZ1sic2FfcmFkaXVzIl0qKjIpCiAgICAKICAgIFpwcmltZSA9IG5wLnplcm9zKCgyICogbnNwb3RzLCBubW9kZXMpKQogICAgCiAgICBmb3IgayBpbiByYW5nZShuc3BvdHMpOgogICAgICAgIHJlZl9jeCA9IHNwb3RzX2RmLmxvY1trLCAicmVmX2N4Il0KICAgICAgICByZWZfY3kgPSBzcG90c19kZi5sb2NbaywgInJlZl9jeSJdCiAgICAgICAgCiAgICAgICAgIyBDYW5vbmljYWwgY2VudGVyIGNvb3JkaW5hdGVzIChvcmlnaW4gYXQgcHVwaWwgY2VudGVyLCBZIHBvaW50aW5nIHVwKQogICAgICAgIHhfYyA9IChyZWZfY3ggLSBzcG90c19kZlsicmVmX2N4Il0ubWVhbigpKSAqIGNmZ1siY2FtZXJhX3BpeHNpemUiXSAvIGNmZ1sicHVwaWxfcmFkaXVzIl0KICAgICAgICB5X2MgPSAtKHJlZl9jeSAtIHNwb3RzX2RmWyJyZWZfY3kiXS5tZWFuKCkpICogY2ZnWyJjYW1lcmFfcGl4c2l6ZSJdIC8gY2ZnWyJwdXBpbF9yYWRpdXMiXQogICAgICAgIAogICAgICAgIGZvciBtX2lkeCwgKGosIG4sIG0pIGluIGVudW1lcmF0ZShtb2Rlcyk6CiAgICAgICAgICAgIHN1bV9kemR4ID0gMC4wCiAgICAgICAgICAgIHN1bV9kemR5ID0gMC4wCiAgICAgICAgICAgIGNvdW50X3B0cyA9IDAKICAgICAgICAgICAgCiAgICAgICAgICAgICMgMkQgYXJlYSBpbnRlZ3JhdGlvbiBvdmVyIHN1Yi1hcGVydHVyZSBkaXNrCiAgICAgICAgICAgIGZvciByX3N0ZXAgaW4gcmFuZ2UoTSk6CiAgICAgICAgICAgICAgICBkeSA9IC1yYmFyICsgMi4wICogcmJhciAqIHJfc3RlcCAvIChNIC0gMSkKICAgICAgICAgICAgICAgIGZvciBjX3N0ZXAgaW4gcmFuZ2UoTSk6CiAgICAgICAgICAgICAgICAgICAgZHggPSAtcmJhciArIDIuMCAqIHJiYXIgKiBjX3N0ZXAgLyAoTSAtIDEpCiAgICAgICAgICAgICAgICAgICAgaWYgZHgqKjIgKyBkeSoqMiA8PSByYmFyKioyOgogICAgICAgICAgICAgICAgICAgICAgICBkemR4LCBkemR5ID0gemVybmlrZV9kZXJpdmF0aXZlcyhuLCBtLCB4X2MgKyBkeCwgeV9jICsgZHkpCiAgICAgICAgICAgICAgICAgICAgICAgIHN1bV9kemR4ICs9IGR6ZHgKICAgICAgICAgICAgICAgICAgICAgICAgc3VtX2R6ZHkgKz0gZHpkeQogICAgICAgICAgICAgICAgICAgICAgICBjb3VudF9wdHMgKz0gMQogICAgICAgICAgICAgICAgICAgICAgICAKICAgICAgICAgICAgYXZnX2R6ZHggPSBzdW1fZHpkeCAvIGNvdW50X3B0cyBpZiBjb3VudF9wdHMgPiAwIGVsc2UgMC4wOwogICAgICAgICAgICBhdmdfZHpkeSA9IHN1bV9kemR5IC8gY291bnRfcHRzIGlmIGNvdW50X3B0cyA+IDAgZWxzZSAwLjA7CiAgICAgICAgICAgIAogICAgICAgICAgICAjIE1hcCB0byBacHJpbWUgZWxlbWVudHMKICAgICAgICAgICAgWnByaW1lW2ssIG1faWR4XSA9IGtrICogYXZnX2R6ZHgKICAgICAgICAgICAgWnByaW1lW2sgKyBuc3BvdHMsIG1faWR4XSA9IC1rayAqIGF2Z19kemR5CiAgICAgICAgICAgIAogICAgIyA0LiBHZW5lcmF0ZSBTYW1wbGVzIHVzaW5nIEFSKDEpIFRlbXBvcmFsIENvcnJlbGF0aW9uCiAgICBwcmludChmIkdlbmVyYXRpbmcge2FyZ3Muc2FtcGxlc30gc2FtcGxlcyAoYXJyYW5nZWQgaW4gdGVtcG9yYWxseSBjb3JyZWxhdGVkIHNlcXVlbmNlcykuLi4iKQogICAgZGlzcGxhY2VtZW50cyA9IG5wLnplcm9zKChhcmdzLnNhbXBsZXMsIDIgKiBuc3BvdHMpKQogICAgY29lZmZpY2llbnRzID0gbnAuemVyb3MoKGFyZ3Muc2FtcGxlcywgbm1vZGVzKSkKICAgIERfcjBfYXJyID0gbnAuemVyb3MoYXJncy5zYW1wbGVzKQogICAgCiAgICAjIFNjYWxlIGZhY3RvciBtYXBwaW5nIGNvZWZmaWNpZW50cyBpbiBtZXRlcnMgdG8gZGlzcGxhY2VtZW50cyBpbiBwaXhlbHMKICAgIHNjYWxlX2ZhY3RvciA9IGNmZ1siZmxlbmd0aCJdIC8gY2ZnWyJjYW1lcmFfcGl4c2l6ZSJdCiAgICAKICAgICMgRGVmaW5lIHNlcXVlbmNlIGxlbmd0aHMgKGRlZmF1bHQgMTAwMCBmcmFtZXMgcGVyIHNlcXVlbmNlIHJlcHJlc2VudGluZyAxIHNlY29uZCBhdCAxMDAwIEh6KQogICAgc2VxX2xlbiA9IDEwMDAKICAgIG5fc2VxID0gYXJncy5zYW1wbGVzIC8vIHNlcV9sZW4KICAgIGlmIG5fc2VxID09IDA6CiAgICAgICAgbl9zZXEgPSAxCiAgICAgICAgc2VxX2xlbiA9IGFyZ3Muc2FtcGxlcwogICAgICAgIAogICAgcHJpbnQoZiIgIENvbmZpZ3VyYXRpb246IHtuX3NlcX0gc2VxdWVuY2VzIG9mIGxlbmd0aCB7c2VxX2xlbn0gZnJhbWVzLiIpCiAgICAKICAgIGZyYW1lX2lkeCA9IDAKICAgIGZvciBzIGluIHJhbmdlKG5fc2VxKToKICAgICAgICAjIFR1cmJ1bGVuY2Ugc3RyZW5ndGggZm9yIHRoaXMgc2VxdWVuY2UKICAgICAgICBEX3IwID0gbnAucmFuZG9tLnVuaWZvcm0oMS4wLCAxMC4wKQogICAgICAgICMgQ29oZXJlbmNlIHRpbWUgZm9yIHRoaXMgc2VxdWVuY2U6IHRhdTAgaW4gcmFuZ2UgMiBtcyB0byAxMCBtcwogICAgICAgIHRhdTAgPSBucC5yYW5kb20udW5pZm9ybSgwLjAwMiwgMC4wMTApCiAgICAgICAgZnMgPSAxMDAwLjAgIyAxMDAwIEh6IGZyYW1lIHJhdGUgKDEgbXMgaW50ZXJ2YWwpCiAgICAgICAgCiAgICAgICAgIyBBUigxKSBjb2VmZmljaWVudDogcmhvID0gZXhwKC1kdCAvIHRhdTApCiAgICAgICAgcmhvID0gbnAuZXhwKC0xLjAgLyAoZnMgKiB0YXUwKSkKICAgICAgICAKICAgICAgICAjIFNjYWxlIGNvdmFyaWFuY2UgYnkgKEQvcjApXig1LzMpCiAgICAgICAgY292ID0gQyAqIChEX3IwICoqICg1LjAgLyAzLjApKQogICAgICAgIHJlZ19jb3YgPSBjb3YgKyBucC5leWUobm1vZGVzKSAqIDFlLTgKICAgICAgICAKICAgICAgICAjIERyYXcgc3RhcnRpbmcgY29lZmZpY2llbnRzIGZvciB0aGUgc2VxdWVuY2UKICAgICAgICBhX3JhZCA9IG5wLnJhbmRvbS5tdWx0aXZhcmlhdGVfbm9ybWFsKG5wLnplcm9zKG5tb2RlcyksIHJlZ19jb3YpCiAgICAgICAgCiAgICAgICAgZm9yIHQgaW4gcmFuZ2Uoc2VxX2xlbik6CiAgICAgICAgICAgIGlmIHQgPiAwOgogICAgICAgICAgICAgICAgIyBBUigxKSBzdGVwOiB1cGRhdGUgWmVybmlrZSBjb2VmZmljaWVudHMgd2l0aCB0ZW1wb3JhbCBjb3JyZWxhdGlvbgogICAgICAgICAgICAgICAgZXBzaWxvbiA9IG5wLnJhbmRvbS5tdWx0aXZhcmlhdGVfbm9ybWFsKG5wLnplcm9zKG5tb2RlcyksIHJlZ19jb3YpCiAgICAgICAgICAgICAgICBhX3JhZCA9IHJobyAqIGFfcmFkICsgbnAuc3FydCgxLjAgLSByaG8gKiByaG8pICogZXBzaWxvbgogICAgICAgICAgICAgICAgCiAgICAgICAgICAgICMgQ29udmVydCB0byBtZXRlcnMgZm9yIHBoeXNpY2FsIHNsb3BlIG1hcHBpbmc6IGFfbSA9IGFfcmFkICogKGxhbWJkYSAvIDJwaSkKICAgICAgICAgICAgYV9tID0gYV9yYWQgKiAoY2ZnWyJ3YXZlbGVuZ3RoIl0gLyAoMi4wICogbnAucGkpKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBDYWxjdWxhdGUgY2xlYW4gZGlzcGxhY2VtZW50cyBpbiBwaXhlbHMKICAgICAgICAgICAgY2xlYW5fZGlzcCA9IFpwcmltZS5kb3QoYV9tKSAqIHNjYWxlX2ZhY3RvcgogICAgICAgICAgICAKICAgICAgICAgICAgIyBBZGQgbm9pc2UKICAgICAgICAgICAgbm9pc2UgPSBucC5yYW5kb20ubm9ybWFsKDAuMCwgYXJncy5ub2lzZSwgMiAqIG5zcG90cykKICAgICAgICAgICAgbm9pc3lfZGlzcCA9IGNsZWFuX2Rpc3AgKyBub2lzZQogICAgICAgICAgICAKICAgICAgICAgICAgZGlzcGxhY2VtZW50c1tmcmFtZV9pZHhdID0gbm9pc3lfZGlzcAogICAgICAgICAgICBjb2VmZmljaWVudHNbZnJhbWVfaWR4XSA9IGFfcmFkCiAgICAgICAgICAgIERfcjBfYXJyW2ZyYW1lX2lkeF0gPSBEX3IwCiAgICAgICAgICAgIAogICAgICAgICAgICBmcmFtZV9pZHggKz0gMQogICAgICAgICAgICBpZiBmcmFtZV9pZHggPj0gYXJncy5zYW1wbGVzOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIAogICAgICAgIHByaW50KGYiICBTZXF1ZW5jZSB7cyArIDE6MDJkfS97bl9zZXE6MDJkfSBjb21wbGV0ZSAoRC9yMD17RF9yMDouMmZ9LCB0YXUwPXt0YXUwKjEwMDAuMDouMmZ9IG1zKS4iKQogICAgICAgIAogICAgIyA1LiBTYXZlIGRhdGFzZXQKICAgIG91dF9kaXIgPSBvcy5wYXRoLmRpcm5hbWUoYXJncy5vdXQpCiAgICBpZiBvdXRfZGlyOgogICAgICAgIG9zLm1ha2VkaXJzKG91dF9kaXIsIGV4aXN0X29rPVRydWUpCiAgICAgICAgCiAgICBucC5zYXZleihhcmdzLm91dCwgCiAgICAgICAgICAgICBkaXNwbGFjZW1lbnRzPWRpc3BsYWNlbWVudHMsIAogICAgICAgICAgICAgY29lZmZpY2llbnRzPWNvZWZmaWNpZW50cywKICAgICAgICAgICAgIERfcjA9RF9yMF9hcnIpCiAgICAgICAgICAgICAKICAgIHByaW50KGYiU3VjY2Vzc2Z1bGx5IHNhdmVkIGRhdGFzZXQgdG8ge2FyZ3Mub3V0fSEiKQogICAgcHJpbnQoZiIgIElucHV0cyBzaGFwZTogIHtkaXNwbGFjZW1lbnRzLnNoYXBlfSIpCiAgICBwcmludChmIiAgVGFyZ2V0cyBzaGFwZToge2NvZWZmaWNpZW50cy5zaGFwZX0iKQoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIGltcG9ydCBzeXMKICAgIGlmIG5vdCAobGVuKHN5cy5hcmd2KSA+IDAgYW5kICgna2VybmVsJyBpbiBzeXMuYXJndlswXSBvciAnLWYnIGluIHN5cy5hcmd2KSk6CiAgICAgICAgbWFpbigpCg=='
with open('generate_dataset.py', 'wb') as f:
    f.write(base64.b64decode(gen_b64))

models_b64 = 'IyBtbC9tb2RlbHMucHkgLSBQeVRvcmNoIE5ldXJhbCBOZXR3b3JrIEFyY2hpdGVjdHVyZXMgZm9yIFdhdmVmcm9udCBSZWNvbnN0cnVjdGlvbgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgpjbGFzcyBXYXZlZnJvbnRNTFAobm4uTW9kdWxlKToKICAgICIiIgogICAgRnVsbHkgQ29ubmVjdGVkIE11bHRpLUxheWVyIFBlcmNlcHRyb24gQmFzZWxpbmUgbW9kZWwuCiAgICBNYXBzIDFEIHZlY3RvciBvZiBkaXNwbGFjZW1lbnRzIGRpcmVjdGx5IHRvIFplcm5pa2UgY29lZmZpY2llbnRzLgogICAgIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5wdXRfZGltPTI1NCwgb3V0cHV0X2RpbT0yMCwgaGlkZGVuX2RpbXM9WzUxMiwgMjU2LCAxMjhdLCBkcm9wb3V0PTAuMSk6CiAgICAgICAgc3VwZXIoV2F2ZWZyb250TUxQLCBzZWxmKS5fX2luaXRfXygpCiAgICAgICAgbGF5ZXJzID0gW10KICAgICAgICBpbl9kaW0gPSBpbnB1dF9kaW0KICAgICAgICBmb3IgaF9kaW0gaW4gaGlkZGVuX2RpbXM6CiAgICAgICAgICAgIGxheWVycy5hcHBlbmQobm4uTGluZWFyKGluX2RpbSwgaF9kaW0pKQogICAgICAgICAgICBsYXllcnMuYXBwZW5kKG5uLkxheWVyTm9ybShoX2RpbSkpCiAgICAgICAgICAgIGxheWVycy5hcHBlbmQobm4uUmVMVSgpKQogICAgICAgICAgICBpZiBkcm9wb3V0ID4gMC4wOgogICAgICAgICAgICAgICAgbGF5ZXJzLmFwcGVuZChubi5Ecm9wb3V0KGRyb3BvdXQpKQogICAgICAgICAgICBpbl9kaW0gPSBoX2RpbQogICAgICAgIGxheWVycy5hcHBlbmQobm4uTGluZWFyKGluX2RpbSwgb3V0cHV0X2RpbSkpCiAgICAgICAgc2VsZi5uZXQgPSBubi5TZXF1ZW50aWFsKCpsYXllcnMpCiAgICAgICAgCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICByZXR1cm4gc2VsZi5uZXQoeCkKCgpjbGFzcyBSZXNCbG9jayhubi5Nb2R1bGUpOgogICAgIiIiCiAgICBTdGFuZGFyZCBSZXNpZHVhbCBibG9jayB3aXRoIEJhdGNoTm9ybSBhbmQgUmVsdSBhY3RpdmF0aW9uLgogICAgIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2hhbm5lbHMpOgogICAgICAgIHN1cGVyKFJlc0Jsb2NrLCBzZWxmKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5jb252MSA9IG5uLkNvbnYyZChjaGFubmVscywgY2hhbm5lbHMsIGtlcm5lbF9zaXplPTMsIHBhZGRpbmc9MSwgYmlhcz1GYWxzZSkKICAgICAgICBzZWxmLmJuMSA9IG5uLkJhdGNoTm9ybTJkKGNoYW5uZWxzKQogICAgICAgIHNlbGYucmVsdSA9IG5uLlJlTFUoKQogICAgICAgIHNlbGYuY29udjIgPSBubi5Db252MmQoY2hhbm5lbHMsIGNoYW5uZWxzLCBrZXJuZWxfc2l6ZT0zLCBwYWRkaW5nPTEsIGJpYXM9RmFsc2UpCiAgICAgICAgc2VsZi5ibjIgPSBubi5CYXRjaE5vcm0yZChjaGFubmVscykKICAgICAgICAKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHJlc2lkdWFsID0geAogICAgICAgIG91dCA9IHNlbGYuY29udjEoeCkKICAgICAgICBvdXQgPSBzZWxmLmJuMShvdXQpCiAgICAgICAgb3V0ID0gc2VsZi5yZWx1KG91dCkKICAgICAgICBvdXQgPSBzZWxmLmNvbnYyKG91dCkKICAgICAgICBvdXQgPSBzZWxmLmJuMihvdXQpCiAgICAgICAgb3V0ICs9IHJlc2lkdWFsCiAgICAgICAgb3V0ID0gc2VsZi5yZWx1KG91dCkKICAgICAgICByZXR1cm4gb3V0CgoKY2xhc3MgV2F2ZWZyb250Q05OKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIFNwYXRpYWwgMkQgQ29udm9sdXRpb25hbCBOZXVyYWwgTmV0d29yayBSZWNvbnN0cnVjdG9yLgogICAgTWFwcyAyLWNoYW5uZWwgMkQgZ3JpZCBvZiBzcG90IGRpc3BsYWNlbWVudHMgKGR4LCBkeSkgdG8gWmVybmlrZSBjb2VmZmljaWVudHMuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBvdXRwdXRfZGltPTIwKToKICAgICAgICBzdXBlcihXYXZlZnJvbnRDTk4sIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbnYxID0gbm4uQ29udjJkKDIsIDMyLCBrZXJuZWxfc2l6ZT0zLCBwYWRkaW5nPTEpCiAgICAgICAgc2VsZi5ibjEgPSBubi5CYXRjaE5vcm0yZCgzMikKICAgICAgICBzZWxmLnJlbHUgPSBubi5SZUxVKCkKICAgICAgICAKICAgICAgICBzZWxmLnJlczEgPSBSZXNCbG9jaygzMikKICAgICAgICBzZWxmLnJlczIgPSBSZXNCbG9jaygzMikKICAgICAgICAKICAgICAgICAjIERvd25zYW1wbGUgdXNpbmcgc3RyaWRlZCBjb252b2x1dGlvbiAoMTN4MTMgLT4gN3g3KQogICAgICAgIHNlbGYuY29udjIgPSBubi5Db252MmQoMzIsIDY0LCBrZXJuZWxfc2l6ZT0zLCBwYWRkaW5nPTEsIHN0cmlkZT0yKQogICAgICAgIHNlbGYuYm4yID0gbm4uQmF0Y2hOb3JtMmQoNjQpCiAgICAgICAgCiAgICAgICAgc2VsZi5yZXMzID0gUmVzQmxvY2soNjQpCiAgICAgICAgCiAgICAgICAgc2VsZi5wb29sID0gbm4uQWRhcHRpdmVBdmdQb29sMmQoKDMsIDMpKQogICAgICAgIHNlbGYuZmMgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoNjQgKiAzICogMywgMTI4KSwKICAgICAgICAgICAgbm4uUmVMVSgpLAogICAgICAgICAgICBubi5Ecm9wb3V0KDAuMSksCiAgICAgICAgICAgIG5uLkxpbmVhcigxMjgsIG91dHB1dF9kaW0pCiAgICAgICAgKQogICAgICAgIAogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgb3V0ID0gc2VsZi5jb252MSh4KQogICAgICAgIG91dCA9IHNlbGYuYm4xKG91dCkKICAgICAgICBvdXQgPSBzZWxmLnJlbHUob3V0KQogICAgICAgIG91dCA9IHNlbGYucmVzMShvdXQpCiAgICAgICAgb3V0ID0gc2VsZi5yZXMyKG91dCkKICAgICAgICBvdXQgPSBzZWxmLmNvbnYyKG91dCkKICAgICAgICBvdXQgPSBzZWxmLmJuMihvdXQpCiAgICAgICAgb3V0ID0gc2VsZi5yZWx1KG91dCkKICAgICAgICBvdXQgPSBzZWxmLnJlczMob3V0KQogICAgICAgIG91dCA9IHNlbGYucG9vbChvdXQpCiAgICAgICAgb3V0ID0gb3V0LnZpZXcob3V0LnNpemUoMCksIC0xKQogICAgICAgIHJldHVybiBzZWxmLmZjKG91dCkK'
with open('models.py', 'wb') as f:
    f.write(base64.b64decode(models_b64))

train_b64 = 'IyBtbC90cmFpbi5weSAtIFRyYWluIFB5VG9yY2ggbW9kZWxzIGZvciBTaGFjay1IYXJ0bWFubiB3YXZlZnJvbnQgcmVjb25zdHJ1Y3Rpb24KaW1wb3J0IG9zCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YXNldCwgRGF0YUxvYWRlciwgcmFuZG9tX3NwbGl0Cgpmcm9tIG1vZGVscyBpbXBvcnQgV2F2ZWZyb250TUxQLCBXYXZlZnJvbnRDTk4KCmNsYXNzIFdhdmVmcm9udERhdGFzZXQoRGF0YXNldCk6CiAgICAiIiIKICAgIERhdGFzZXQgbG9hZGVyIG1hcHBpbmcgMUQgZGlzcGxhY2VtZW50cyB0byBaZXJuaWtlIGNvZWZmaWNpZW50cy4KICAgIElmIG1vZGUgaXMgJ2NubicsIGhhbmRsZXMgc3BhdGlhbCBtYXBwaW5nIG9mIHRoZSAxMjcgc3BvdHMgdG8gYSAyRCBncmlkLgogICAgIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgZGF0YXNldF9wYXRoLCBzcG90c19jc3ZfcGF0aCwgbW9kZWxfdHlwZT0nbWxwJyk6CiAgICAgICAgc2VsZi5tb2RlbF90eXBlID0gbW9kZWxfdHlwZQogICAgICAgIAogICAgICAgICMgTG9hZCBucHogZGF0YXNldAogICAgICAgIGRhdGEgPSBucC5sb2FkKGRhdGFzZXRfcGF0aCkKICAgICAgICBzZWxmLmRpc3BsYWNlbWVudHMgPSB0b3JjaC50ZW5zb3IoZGF0YVsnZGlzcGxhY2VtZW50cyddLCBkdHlwZT10b3JjaC5mbG9hdDMyKQogICAgICAgIHNlbGYuY29lZmZpY2llbnRzID0gdG9yY2gudGVuc29yKGRhdGFbJ2NvZWZmaWNpZW50cyddLCBkdHlwZT10b3JjaC5mbG9hdDMyKQogICAgICAgIHNlbGYubl9zYW1wbGVzID0gbGVuKHNlbGYuZGlzcGxhY2VtZW50cykKICAgICAgICBzZWxmLm5zcG90cyA9IHNlbGYuZGlzcGxhY2VtZW50cy5zaGFwZVsxXSAvLyAyCiAgICAgICAgCiAgICAgICAgIyBMb2FkIHNwb3QgY29vcmRpbmF0ZXMgZm9yIHNwYXRpYWwgQ05OIG1hcHBpbmcKICAgICAgICBzcG90c19kZiA9IHBkLnJlYWRfY3N2KHNwb3RzX2Nzdl9wYXRoKQogICAgICAgIAogICAgICAgICMgRXN0aW1hdGUgcGl0Y2ggYW5kIGNlbnRlcgogICAgICAgIG1lYW5fY3ggPSBzcG90c19kZlsicmVmX2N4Il0ubWVhbigpCiAgICAgICAgbWVhbl9jeSA9IHNwb3RzX2RmWyJyZWZfY3kiXS5tZWFuKCkKICAgICAgICAKICAgICAgICAjIEFwcHJveGltYXRlIHBpdGNoIGluIHBpeGVscyAofjQwLjEgcHgpCiAgICAgICAgIyBGaW5kIG1pbmltdW0gZGlzdGFuY2UgdG8gbmVpZ2hib3IgZm9yIHBpdGNoCiAgICAgICAgZGlzdHMgPSBbXQogICAgICAgIGZvciBpIGluIHJhbmdlKGxlbihzcG90c19kZikpOgogICAgICAgICAgICBkeCA9IHNwb3RzX2RmWyJyZWZfY3giXS52YWx1ZXMgLSBzcG90c19kZlsicmVmX2N4Il0udmFsdWVzW2ldCiAgICAgICAgICAgIGR5ID0gc3BvdHNfZGZbInJlZl9jeSJdLnZhbHVlcyAtIHNwb3RzX2RmWyJyZWZfY3kiXS52YWx1ZXNbaV0KICAgICAgICAgICAgZCA9IG5wLmh5cG90KGR4LCBkeSkKICAgICAgICAgICAgZCA9IGRbZCA+IDFlLTNdCiAgICAgICAgICAgIGlmIGxlbihkKSA+IDA6CiAgICAgICAgICAgICAgICBkaXN0cy5hcHBlbmQoZC5taW4oKSkKICAgICAgICBwaXRjaF9weCA9IG5wLm1lYW4oZGlzdHMpIGlmIGxlbihkaXN0cykgPiAwIGVsc2UgNDAuMQogICAgICAgIAogICAgICAgICMgTWFwIHNwb3RzIHRvIGludGVnZXIgZ3JpZCBpbmRpY2VzICh1LCB2KQogICAgICAgIHVfY29vcmRzID0gbnAucm91bmQoKHNwb3RzX2RmWyJyZWZfY3giXS52YWx1ZXMgLSBtZWFuX2N4KSAvIHBpdGNoX3B4KS5hc3R5cGUoaW50KQogICAgICAgIHZfY29vcmRzID0gbnAucm91bmQoKHNwb3RzX2RmWyJyZWZfY3kiXS52YWx1ZXMgLSBtZWFuX2N5KSAvIHBpdGNoX3B4KS5hc3R5cGUoaW50KQogICAgICAgIAogICAgICAgIHVfbWluLCB1X21heCA9IHVfY29vcmRzLm1pbigpLCB1X2Nvb3Jkcy5tYXgoKQogICAgICAgIHZfbWluLCB2X21heCA9IHZfY29vcmRzLm1pbigpLCB2X2Nvb3Jkcy5tYXgoKQogICAgICAgIAogICAgICAgIHNlbGYuZ3JpZF93ID0gaW50KHVfbWF4IC0gdV9taW4gKyAxKQogICAgICAgIHNlbGYuZ3JpZF9oID0gaW50KHZfbWF4IC0gdl9taW4gKyAxKQogICAgICAgIHNlbGYudV9vZmZzZXQgPSBpbnQoLXVfbWluKQogICAgICAgIHNlbGYudl9vZmZzZXQgPSBpbnQoLXZfbWluKQogICAgICAgIAogICAgICAgIHNlbGYuc3BvdF9ncmlkX2Nvb3JkcyA9IGxpc3QoemlwKHVfY29vcmRzLCB2X2Nvb3JkcykpCiAgICAgICAgCiAgICBkZWYgX19sZW5fXyhzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5uX3NhbXBsZXMKICAgICAgICAKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpZHgpOgogICAgICAgIGRpc3AgPSBzZWxmLmRpc3BsYWNlbWVudHNbaWR4XQogICAgICAgIGNvZWZmID0gc2VsZi5jb2VmZmljaWVudHNbaWR4XQogICAgICAgIAogICAgICAgIGlmIHNlbGYubW9kZWxfdHlwZSA9PSAnbWxwJzoKICAgICAgICAgICAgcmV0dXJuIGRpc3AsIGNvZWZmCiAgICAgICAgZWxzZTogIyBjbm4KICAgICAgICAgICAgIyBBcnJhbmdlIDFEIGRpc3BsYWNlbWVudHMgaW50byAyLWNoYW5uZWwgMkQgc3BhdGlhbCBncmlkIChkeCwgZHkpCiAgICAgICAgICAgIGdyaWQgPSB0b3JjaC56ZXJvcygoMiwgc2VsZi5ncmlkX2gsIHNlbGYuZ3JpZF93KSwgZHR5cGU9dG9yY2guZmxvYXQzMikKICAgICAgICAgICAgZm9yIGsgaW4gcmFuZ2Uoc2VsZi5uc3BvdHMpOgogICAgICAgICAgICAgICAgdSwgdiA9IHNlbGYuc3BvdF9ncmlkX2Nvb3Jkc1trXQogICAgICAgICAgICAgICAgcm93ID0gdiArIHNlbGYudl9vZmZzZXQKICAgICAgICAgICAgICAgIGNvbCA9IHUgKyBzZWxmLnVfb2Zmc2V0CiAgICAgICAgICAgICAgICBncmlkWzAsIHJvdywgY29sXSA9IGRpc3Bba10KICAgICAgICAgICAgICAgIGdyaWRbMSwgcm93LCBjb2xdID0gZGlzcFtrICsgc2VsZi5uc3BvdHNdCiAgICAgICAgICAgIHJldHVybiBncmlkLCBjb2VmZgoKZGVmIHRyYWluX2Vwb2NoKG1vZGVsLCBkYXRhbG9hZGVyLCBjcml0ZXJpb24sIG9wdGltaXplciwgZGV2aWNlKToKICAgIG1vZGVsLnRyYWluKCkKICAgIHJ1bm5pbmdfbG9zcyA9IDAuMAogICAgZm9yIGlucHV0cywgdGFyZ2V0cyBpbiBkYXRhbG9hZGVyOgogICAgICAgIGlucHV0cywgdGFyZ2V0cyA9IGlucHV0cy50byhkZXZpY2UpLCB0YXJnZXRzLnRvKGRldmljZSkKICAgICAgICAKICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKICAgICAgICBvdXRwdXRzID0gbW9kZWwoaW5wdXRzKQogICAgICAgIGxvc3MgPSBjcml0ZXJpb24ob3V0cHV0cywgdGFyZ2V0cykKICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICBvcHRpbWl6ZXIuc3RlcCgpCiAgICAgICAgCiAgICAgICAgcnVubmluZ19sb3NzICs9IGxvc3MuaXRlbSgpICogaW5wdXRzLnNpemUoMCkKICAgIHJldHVybiBydW5uaW5nX2xvc3MgLyBsZW4oZGF0YWxvYWRlci5kYXRhc2V0KQoKZGVmIGV2YWx1YXRlKG1vZGVsLCBkYXRhbG9hZGVyLCBjcml0ZXJpb24sIGRldmljZSk6CiAgICBtb2RlbC5ldmFsKCkKICAgIHJ1bm5pbmdfbG9zcyA9IDAuMAogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgZm9yIGlucHV0cywgdGFyZ2V0cyBpbiBkYXRhbG9hZGVyOgogICAgICAgICAgICBpbnB1dHMsIHRhcmdldHMgPSBpbnB1dHMudG8oZGV2aWNlKSwgdGFyZ2V0cy50byhkZXZpY2UpCiAgICAgICAgICAgIG91dHB1dHMgPSBtb2RlbChpbnB1dHMpCiAgICAgICAgICAgIGxvc3MgPSBjcml0ZXJpb24ob3V0cHV0cywgdGFyZ2V0cykKICAgICAgICAgICAgcnVubmluZ19sb3NzICs9IGxvc3MuaXRlbSgpICogaW5wdXRzLnNpemUoMCkKICAgIHJldHVybiBydW5uaW5nX2xvc3MgLyBsZW4oZGF0YWxvYWRlci5kYXRhc2V0KQoKZGVmIG1haW4oKToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJUcmFpbiBTaGFjay1IYXJ0bWFubiBuZXVyYWwgbmV0d29yayByZWNvbnN0cnVjdG9yIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbW9kZWwiLCB0eXBlPXN0ciwgZGVmYXVsdD0ibWxwIiwgY2hvaWNlcz1bIm1scCIsICJjbm4iXSwgaGVscD0iTW9kZWwgdHlwZSB0byB0cmFpbiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWVwb2NocyIsIHR5cGU9aW50LCBkZWZhdWx0PTUwLCBoZWxwPSJOdW1iZXIgb2YgZXBvY2hzIHRvIHRyYWluIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYmF0Y2hfc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTY0LCBoZWxwPSJCYXRjaCBzaXplIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbHIiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTFlLTMsIGhlbHA9IkxlYXJuaW5nIHJhdGUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhc2V0IiwgdHlwZT1zdHIsIGRlZmF1bHQ9ImRhdGFfYWkvZGF0YXNldC5ucHoiLCBoZWxwPSJEYXRhc2V0IHBhdGgiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRfZGlyIiwgdHlwZT1zdHIsIGRlZmF1bHQ9Im1sX2NoZWNrcG9pbnRzIiwgaGVscD0iT3V0cHV0IGNoZWNrcG9pbnRzIGRpcmVjdG9yeSIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIHByaW50KGYiVXNpbmcgZXhlY3V0aW9uIGRldmljZToge2RldmljZX0iKQogICAgCiAgICAjIDEuIExvYWQgZGF0YXNldAogICAgc3BvdHNfY3N2ID0gInJlc3VsdHMvcmVmZXJlbmNlX2NlbnRyb2lkc19jLmNzdiIKICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhzcG90c19jc3YpOgogICAgICAgIHByaW50KGYiRXJyb3I6IHtzcG90c19jc3Z9IG5vdCBmb3VuZC4gUGxlYXNlIGNhbGlicmF0ZSBmaXJzdCB1c2luZyBDIHBpcGVsaW5lLiIpCiAgICAgICAgcmV0dXJuCiAgICAgICAgCiAgICBwcmludChmIkxvYWRpbmcgZGF0YXNldCBmcm9tIHthcmdzLmRhdGFzZXR9Li4uIikKICAgIGZ1bGxfZGF0YXNldCA9IFdhdmVmcm9udERhdGFzZXQoYXJncy5kYXRhc2V0LCBzcG90c19jc3YsIG1vZGVsX3R5cGU9YXJncy5tb2RlbCkKICAgIAogICAgIyBUcmFpbiAvIFZhbCAvIFRlc3Qgc3BsaXQgKDgwJSAvIDEwJSAvIDEwJSkKICAgIHRvdGFsX2xlbiA9IGxlbihmdWxsX2RhdGFzZXQpCiAgICB0cmFpbl9sZW4gPSBpbnQoMC44ICogdG90YWxfbGVuKQogICAgdmFsX2xlbiA9IGludCgwLjEgKiB0b3RhbF9sZW4pCiAgICB0ZXN0X2xlbiA9IHRvdGFsX2xlbiAtIHRyYWluX2xlbiAtIHZhbF9sZW4KICAgIAogICAgdHJhaW5fc2V0LCB2YWxfc2V0LCB0ZXN0X3NldCA9IHJhbmRvbV9zcGxpdCgKICAgICAgICBmdWxsX2RhdGFzZXQsIFt0cmFpbl9sZW4sIHZhbF9sZW4sIHRlc3RfbGVuXSwgCiAgICAgICAgZ2VuZXJhdG9yPXRvcmNoLkdlbmVyYXRvcigpLm1hbnVhbF9zZWVkKDQyKQogICAgKQogICAgCiAgICB0cmFpbl9sb2FkZXIgPSBEYXRhTG9hZGVyKHRyYWluX3NldCwgYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoX3NpemUsIHNodWZmbGU9VHJ1ZSkKICAgIHZhbF9sb2FkZXIgPSBEYXRhTG9hZGVyKHZhbF9zZXQsIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplLCBzaHVmZmxlPUZhbHNlKQogICAgdGVzdF9sb2FkZXIgPSBEYXRhTG9hZGVyKHRlc3Rfc2V0LCBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwgc2h1ZmZsZT1GYWxzZSkKICAgIAogICAgcHJpbnQoZiJEYXRhc2V0IHNwbGl0OiBUcmFpbj17dHJhaW5fbGVufSwgVmFsPXt2YWxfbGVufSwgVGVzdD17dGVzdF9sZW59IikKICAgIAogICAgIyAyLiBTZXR1cCBtb2RlbAogICAgaWYgYXJncy5tb2RlbCA9PSAibWxwIjoKICAgICAgICBtb2RlbCA9IFdhdmVmcm9udE1MUChpbnB1dF9kaW09ZnVsbF9kYXRhc2V0Lm5zcG90cyAqIDIsIG91dHB1dF9kaW09ZnVsbF9kYXRhc2V0LmNvZWZmaWNpZW50cy5zaGFwZVsxXSkKICAgIGVsc2U6ICMgY25uCiAgICAgICAgbW9kZWwgPSBXYXZlZnJvbnRDTk4ob3V0cHV0X2RpbT1mdWxsX2RhdGFzZXQuY29lZmZpY2llbnRzLnNoYXBlWzFdKQogICAgICAgIAogICAgbW9kZWwgPSBtb2RlbC50byhkZXZpY2UpCiAgICBwcmludChtb2RlbCkKICAgIAogICAgIyBMb3NzICYgT3B0aW1pemVyCiAgICBjcml0ZXJpb24gPSBubi5NU0VMb3NzKCkKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW1XKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9YXJncy5sciwgd2VpZ2h0X2RlY2F5PTFlLTQpCiAgICBzY2hlZHVsZXIgPSB0b3JjaC5vcHRpbS5scl9zY2hlZHVsZXIuQ29zaW5lQW5uZWFsaW5nTFIob3B0aW1pemVyLCBUX21heD1hcmdzLmVwb2NocykKICAgIAogICAgIyAzLiBUcmFpbmluZyBMb29wCiAgICBvcy5tYWtlZGlycyhhcmdzLm91dF9kaXIsIGV4aXN0X29rPVRydWUpCiAgICBiZXN0X3ZhbF9sb3NzID0gZmxvYXQoImluZiIpCiAgICAKICAgIHByaW50KCJcblN0YXJ0aW5nIFRyYWluaW5nLi4uIikKICAgIGZvciBlcG9jaCBpbiByYW5nZShhcmdzLmVwb2Nocyk6CiAgICAgICAgdHJhaW5fbG9zcyA9IHRyYWluX2Vwb2NoKG1vZGVsLCB0cmFpbl9sb2FkZXIsIGNyaXRlcmlvbiwgb3B0aW1pemVyLCBkZXZpY2UpCiAgICAgICAgdmFsX2xvc3MgPSBldmFsdWF0ZShtb2RlbCwgdmFsX2xvYWRlciwgY3JpdGVyaW9uLCBkZXZpY2UpCiAgICAgICAgc2NoZWR1bGVyLnN0ZXAoKQogICAgICAgIAogICAgICAgIHByaW50KGYiRXBvY2gge2Vwb2NoKzE6MDJkfS97YXJncy5lcG9jaHM6MDJkfSB8IFRyYWluIE1TRToge3RyYWluX2xvc3M6LjZmfSB8IFZhbCBNU0U6IHt2YWxfbG9zczouNmZ9IikKICAgICAgICAKICAgICAgICAjIFNhdmUgYmVzdCBjaGVja3BvaW50CiAgICAgICAgaWYgdmFsX2xvc3MgPCBiZXN0X3ZhbF9sb3NzOgogICAgICAgICAgICBiZXN0X3ZhbF9sb3NzID0gdmFsX2xvc3MKICAgICAgICAgICAgY2hlY2twb2ludF9wYXRoID0gb3MucGF0aC5qb2luKGFyZ3Mub3V0X2RpciwgZiJiZXN0X3thcmdzLm1vZGVsfS5wdCIpCiAgICAgICAgICAgIHRvcmNoLnNhdmUoewogICAgICAgICAgICAgICAgJ2Vwb2NoJzogZXBvY2gsCiAgICAgICAgICAgICAgICAnbW9kZWxfc3RhdGVfZGljdCc6IG1vZGVsLnN0YXRlX2RpY3QoKSwKICAgICAgICAgICAgICAgICdvcHRpbWl6ZXJfc3RhdGVfZGljdCc6IG9wdGltaXplci5zdGF0ZV9kaWN0KCksCiAgICAgICAgICAgICAgICAndmFsX2xvc3MnOiB2YWxfbG9zcywKICAgICAgICAgICAgfSwgY2hlY2twb2ludF9wYXRoKQogICAgICAgICAgICAKICAgICMgNC4gRmluYWwgRXZhbHVhdGlvbgogICAgY2hlY2twb2ludF9wYXRoID0gb3MucGF0aC5qb2luKGFyZ3Mub3V0X2RpciwgZiJiZXN0X3thcmdzLm1vZGVsfS5wdCIpCiAgICBjaGVja3BvaW50ID0gdG9yY2gubG9hZChjaGVja3BvaW50X3BhdGgsIG1hcF9sb2NhdGlvbj1kZXZpY2UpCiAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoY2hlY2twb2ludFsnbW9kZWxfc3RhdGVfZGljdCddKQogICAgCiAgICB0ZXN0X2xvc3MgPSBldmFsdWF0ZShtb2RlbCwgdGVzdF9sb2FkZXIsIGNyaXRlcmlvbiwgZGV2aWNlKQogICAgcHJpbnQoZiJcbkZpbmFsIFRlc3QgTVNFIExvc3Mgb24gQmVzdCBDaGVja3BvaW50OiB7dGVzdF9sb3NzOi42Zn0iKQogICAgCiAgICAjIENvbXBhcmUgd2l0aCBDIFJlY29uc3RydWN0b3Igb24gNSB0ZXN0IHNhbXBsZXMKICAgIG1vZGVsLmV2YWwoKQogICAgcHJpbnQoIlxuVmlzdWFsIHNwb3QgY2hlY2s6IFRydWUgdnMgUHJlZGljdGVkIFplcm5pa2UgY29lZmZzIChmaXJzdCA1IG1vZGVzIG9mIHNhbXBsZSAxKToiKQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgaW5wdXRzLCB0YXJnZXRzID0gbmV4dChpdGVyKHRlc3RfbG9hZGVyKSkKICAgICAgICBpbnB1dHMsIHRhcmdldHMgPSBpbnB1dHMudG8oZGV2aWNlKSwgdGFyZ2V0cy50byhkZXZpY2UpCiAgICAgICAgcHJlZHMgPSBtb2RlbChpbnB1dHMpCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UoNSk6CiAgICAgICAgICAgIHByaW50KGYiICBTYW1wbGUge2krMX06IFRydWU9e1tmbG9hdCh4KSBmb3IgeCBpbiB0YXJnZXRzW2ldWzo1XV19IHwgUHJlZD17W2Zsb2F0KHgpIGZvciB4IGluIHByZWRzW2ldWzo1XV19IikKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBpbXBvcnQgc3lzCiAgICBpZiBub3QgKGxlbihzeXMuYXJndikgPiAwIGFuZCAoJ2tlcm5lbCcgaW4gc3lzLmFyZ3ZbMF0gb3IgJy1mJyBpbiBzeXMuYXJndikpOgogICAgICAgIG1haW4oKQo='
with open('train.py', 'wb') as f:
    f.write(base64.b64decode(train_b64))

print('All files decoded and written to disk successfully!')

## Step 2: Generate Massive Dataset
Generate 50,000 samples of temporally correlated synthetic Kolmogorov turbulence.

In [ ]:
!python generate_dataset.py --samples 50000 --out data_ai/dataset.npz --noise 0.1

## Step 3: Train WavefrontMLP Reconstructor
Train the MLP baseline for 30 epochs.

In [ ]:
!python train.py --model mlp --epochs 30 --batch_size 128 --lr 1e-3

## Step 4: Train WavefrontCNN Reconstructor
Train the ResNet-based CNN reconstructor for 30 epochs.

In [ ]:
!python train.py --model cnn --epochs 30 --batch_size 128 --lr 1e-3